# Prism：Jupyter 多核通用对战
选择导出的模型JSON、对手和双座局数，后台并行运行；自动保存完整Markdown/HTML及原始回放。

**这是通用开发评测，不能替代M63正式验收。** 无systemd、SSH、root、GPU或PyTorch要求。8核机器默认6个对局worker、每worker计算库1线程；容器CPU配额较小会自动降低默认并发。进程树限额为软监控，并非独占cgroup硬隔离。

先完成代码和权重导入，再运行准备/启动单元格。重新连接Notebook时只需恢复变量并运行“状态”，不要再次创建任务。平台关机或回收仍会中断后台进程。


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, hashlib, zipfile, urllib.request, platform

REPO_URL = "https://github.com/zxjAiLun/effective-splendor.git"
REPO_REF = "main"  # 发布含M64的提交后，建议填写那个确切commit。
existing = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"Cargo.toml").is_file() and (p/".git").exists()), None)
REPO_ROOT = existing or (Path.cwd()/"effective-splendor")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_ROOT, check=True)
print("代码目录：", REPO_ROOT)
print(subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip())


## 代码版本 / 可选源码包
本轮M64目前只在本机完成，尚未提交/push：clone旧提交不会自动出现新Python工具或最新Rust接口。正式发布后可直接clone其commit；发布前可给这个clone应用已生成的**源码包**（不含权重/回放/密钥）。填写可访问的HTTPS下载URL及随包提供的SHA256，或把包放在Notebook文件目录后填写路径。源包URL不能凭空生成。


In [ ]:
SOURCE_BUNDLE_URL = ""
SOURCE_BUNDLE_SHA256 = ""
SOURCE_BUNDLE_PATH = REPO_ROOT/"local-artifacts"/"prism-notebook-source.zip"
required = ["scripts/prism_notebook_arena.py", "scripts/prism_notebook_bundle.py", "scripts/prism_ppo_fixed_eval.py"]
if SOURCE_BUNDLE_URL:
    if len(SOURCE_BUNDLE_SHA256) != 64: raise ValueError("源码包需要SHA256")
    SOURCE_BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(SOURCE_BUNDLE_URL, timeout=60) as response, SOURCE_BUNDLE_PATH.open("wb") as out:
        shutil.copyfileobj(response, out)
if SOURCE_BUNDLE_PATH.exists():
    if hashlib.sha256(SOURCE_BUNDLE_PATH.read_bytes()).hexdigest() != SOURCE_BUNDLE_SHA256:
        raise ValueError("源码包SHA256不匹配")
    with zipfile.ZipFile(SOURCE_BUNDLE_PATH) as z:
        manifest=json.loads(z.read("manifest.json"))
        if manifest["format"] != "prism-notebook-source-bundle-v1": raise ValueError("不是源码包")
        if len(z.namelist()) != len(set(z.namelist())) or set(z.namelist()) != set(manifest["files"])|{"manifest.json"}: raise ValueError("源码包目录错误")
        if sum(x.file_size for x in z.infolist()) > 256*1024**2: raise ValueError("源码包过大")
        for name,digest in manifest["files"].items():
            p=Path(name);target=(REPO_ROOT/p).resolve()
            if p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name or not target.is_relative_to(REPO_ROOT.resolve()): raise ValueError("不安全的路径")
            if name not in ("Cargo.toml","Cargo.lock") and p.parts[0] not in ("crates","scripts","notebooks","docs"): raise ValueError("路径超出源码范围")
            if hashlib.sha256(z.read(name)).hexdigest() != digest: raise ValueError("源码成员SHA不符")
        for name in manifest["files"]:
            p=REPO_ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
        (REPO_ROOT/"local-artifacts"/"source-overlay-manifest.json").write_text(json.dumps(manifest,indent=2))
if not all((REPO_ROOT/p).is_file() for p in required):
    raise RuntimeError("这个clone还没有M64工具：请切换已发布的M64 commit，或填写源码包来源/SHA。")


## 隔离Python依赖
使用独立venv，保持Notebook原有训练/模型环境。需要Python3.10+；当前实测环境见M64文档。该步骤只安装numpy/psutil/Markdown，不安装torch。

In [ ]:
if sys.version_info < (3,10): raise RuntimeError("请选择Python3.10或更高的Notebook内核")
VENV = REPO_ROOT/"local-artifacts"/"notebook-venv"
ARENA_PYTHON = VENV/("Scripts/python.exe" if os.name=="nt" else "bin/python")
if not ARENA_PYTHON.exists(): subprocess.run([sys.executable,"-m","venv",str(VENV)],check=True)
subprocess.run([str(ARENA_PYTHON),"-m","pip","install","-r",str(REPO_ROOT/"notebooks/requirements-prism-arena.txt")],check=True)

def arena_rpc(function, **kwargs):
    module,method = function.split(":")
    if module not in ("prism_notebook_arena", "prism_notebook_bundle"): raise ValueError("不支持的工具模块")
    script = "import importlib,json,sys; p=json.load(sys.stdin); m=importlib.import_module('scripts.'+p['module']); result=getattr(m,p['method'])(**p['kwargs']); print(json.dumps(result,default=str))"
    env=dict(os.environ)
    for key in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","RAYON_NUM_THREADS"):env[key]="1"
    result=subprocess.run([str(ARENA_PYTHON),"-B","-c",script],cwd=REPO_ROOT,env=env,
        input=json.dumps({"module":module,"method":method,"kwargs":kwargs}),capture_output=True,text=True)
    if result.returncode: raise RuntimeError(result.stderr or result.stdout)
    return json.loads(result.stdout)
print(json.dumps(arena_rpc("prism_notebook_arena:resource_profile"),ensure_ascii=False,indent=2))


## 权重输入
权重按项目约定不加入Git，clone源码不能获得本轮候选。可选用独立模型包，或把已有导出JSON放在下面指定的路径。HTTPS下载必须填写SHA；只接受`prism-weights-v1` JSON，不接受恢复用的torch checkpoint。

模型包中版本名必须完整；未发布候选会保留candidate标记。提供的示例包可包含Prism Balance v0.1.0、Prism v0.1.1及本轮未发布Prism Beta v0.1.1 candidate，任选，导入不代表注册/发布。

In [ ]:
INPUT_DIR = REPO_ROOT/"local-artifacts"/"notebook-inputs"
MODEL_BUNDLE_URL = ""
MODEL_BUNDLE_SHA256 = ""
MODEL_BUNDLE_PATH = INPUT_DIR/"models.zip"
if MODEL_BUNDLE_URL:
    arena_rpc("prism_notebook_bundle:fetch",url=MODEL_BUNDLE_URL,path=str(MODEL_BUNDLE_PATH),expected_sha256=MODEL_BUNDLE_SHA256)
if MODEL_BUNDLE_PATH.exists():
    model_manifest=arena_rpc("prism_notebook_bundle:unpack",bundle=str(MODEL_BUNDLE_PATH),destination=str(INPUT_DIR),expected_sha256=MODEL_BUNDLE_SHA256)
    print(json.dumps(model_manifest["models"],ensure_ascii=False,indent=2))
else:
    print("未导入模型包；请在选模型单元格填写已有导出JSON路径。")


## 编译Linux原生引擎与回放指标工具
默认使用隔离build目录，避免旧二进制混入。已有Cargo会直接使用；缺少Cargo时可在本用户目录安装官方Rust工具链，无sudo。需要系统中有C编译器（例如gcc/clang）；若平台不提供，先选带编译工具的镜像。本步可能需数分钟及网络访问，不计入对局耗时。

In [ ]:
ENV = dict(os.environ)
if not shutil.which("cargo"):
    if platform.system() != "Linux": raise RuntimeError("没有Cargo：请先安装Rust工具链")
    arch={"x86_64":"x86_64","aarch64":"aarch64"}.get(platform.machine())
    if not arch: raise RuntimeError("请手动安装该架构的Rust")
    tools=REPO_ROOT/"local-artifacts"/"notebook-tools";tools.mkdir(parents=True,exist_ok=True)
    init=tools/"rustup-init"
    url=f"https://static.rust-lang.org/rustup/dist/{arch}-unknown-linux-gnu/rustup-init"
    digest=urllib.request.urlopen(url+".sha256",timeout=60).read().decode().split()[0]
    with urllib.request.urlopen(url,timeout=60) as response,init.open("wb") as out: shutil.copyfileobj(response,out)
    if hashlib.sha256(init.read_bytes()).hexdigest()!=digest: raise ValueError("官方Rust安装器SHA不符")
    init.chmod(0o700)
    ENV["RUSTUP_HOME"]=str(tools/"rustup");ENV["CARGO_HOME"]=str(tools/"cargo")
    subprocess.run([str(init),"-y","--no-modify-path","--profile","minimal","--default-toolchain","1.90.0"],env=ENV,check=True)
    ENV["PATH"]=str(tools/"cargo/bin")+os.pathsep+ENV["PATH"]
if platform.system()=="Linux" and not (shutil.which("cc",path=ENV["PATH"]) or shutil.which("gcc",path=ENV["PATH"]) or shutil.which("clang",path=ENV["PATH"])):
    raise RuntimeError("此Notebook镜像没有C编译器；需使用带gcc/clang的镜像")
BUILD_DIR=REPO_ROOT/"local-artifacts"/"notebook-build";ENV["CARGO_TARGET_DIR"]=str(BUILD_DIR)
profile=arena_rpc("prism_notebook_arena:resource_profile")
subprocess.run(["cargo","build","--release","--locked","-p","splendor-cli","--bin","splendor","--example","s4_batch_metrics","--jobs",str(min(6,profile["effective_whole_cores"]))],cwd=REPO_ROOT,env=ENV,check=True)
ENGINE=BUILD_DIR/"release"/("splendor.exe" if os.name=="nt" else "splendor")
METRICS=BUILD_DIR/"release/examples"/("s4_batch_metrics.exe" if os.name=="nt" else "s4_batch_metrics")
print(subprocess.run([str(ENGINE),"version"],capture_output=True,text=True,check=True).stdout)


## 选择模型、对手、局数
下面只是可编辑示例：`MODELS`的第一项是匹配基线，候选采样T=1；对手可greedy或sample。`GAMES_PER_PAIR`是**每个模型×每个对手的物理局数**，必须偶数；种子数为一半。先用4局/对做接口检查，再显式创建新的任务名做大批量。示例默认4局；不会自动扩大为正式验收。

In [ ]:
MODEL_SPECS = [
    {"id":"baseline", "weights":str(INPUT_DIR/"models/balance.json"), "display_name":"Prism Balance v0.1.0"},
    {"id":"candidate", "weights":str(INPUT_DIR/"models/beta-m63.json"), "display_name":"Prism Beta v0.1.1 candidate（未发布）"},
]
OPPONENT_SPECS = [ {"id":"heuristic", "kind":"heuristic"} ]
# 可加：{"id":"general", "kind":"prism", "weights":str(INPUT_DIR/"models/general.json"), "display_name":"Prism v0.1.1", "mode":"sample"}
GAMES_PER_PAIR = 4  # 正式开发批量可自行设为256/512/1024；这是开发测量，不自动晋级。
SEED_START = 64000000
ACTION_SEED = 64010001
WORKERS = None  # 默认按有效CPU留资源；8核通常6。也可显式设8，不能超过容器有效分配。
MEMORY_GIB = None  # 自动预留内存；8核32GiB可按实际可用量设24。
RUN_NAME = "notebook-smoke-001"  # 每次新配置必须改名；原任务不可覆盖。
RUN_DIR = REPO_ROOT/"local-artifacts"/"notebook-arena"/RUN_NAME
if GAMES_PER_PAIR < 2 or GAMES_PER_PAIR % 2: raise ValueError("双座局数须为正偶数")
candidates={}
for spec in MODEL_SPECS:
    if not Path(spec["weights"]).is_file(): raise FileNotFoundError("模型权重缺失：请先导入模型包或填写真实JSON路径")
    candidates[spec["id"]]=arena_rpc("prism_notebook_arena:model",weights=spec["weights"],display_name=spec["display_name"],engine=str(ENGINE),action_seed=ACTION_SEED)
opponents={}
for spec in OPPONENT_SPECS:
    opponents[spec["id"]]=arena_rpc("prism_notebook_arena:heuristic",engine=str(ENGINE)) if spec["kind"]=="heuristic" else arena_rpc("prism_notebook_arena:model",weights=spec["weights"],display_name=spec["display_name"],engine=str(ENGINE),mode=spec.get("mode","greedy"),action_seed=ACTION_SEED)
print("总物理局数：",len(candidates)*len(opponents)*GAMES_PER_PAIR)


## 固定配置（零局） / 启动后台任务
准备后会冻结种子、配置、源文件/模型/二进制hash；改参数须换任务名。启动单元格返回PID后即可另做工作。不要在两个主机上同时派发同一正式网格；本工具不管理M63正式网格。

In [ ]:
plan=arena_rpc("prism_notebook_arena:prepare",work=str(RUN_DIR),engine=str(ENGINE),metrics_program=str(METRICS),
    candidates=candidates,opponents=opponents,seeds=list(range(SEED_START,SEED_START+GAMES_PER_PAIR//2)),
    workers=WORKERS,action_seed=ACTION_SEED,memory_gib=MEMORY_GIB,wall_hours=8,cpu_hours=24,storage_gib=32)
print("配置已固定，0新局：",plan["total_games"],"局 /",plan["limits"]["workers"],"workers")


In [ ]:
owner=arena_rpc("prism_notebook_arena:start",work=str(RUN_DIR))
print("后台已启动：",owner)


## 查看进度 / 预计剩余时间
可反复运行此单元格，或重新连接后恢复变量再运行。初期只有几个分片完成时ETA不稳定；报告/回放验核时间另算。

In [ ]:
from IPython.display import display, HTML, FileLink
s=arena_rpc("prism_notebook_arena:status",work=str(RUN_DIR))
print(json.dumps(s,ensure_ascii=False,indent=2))
if s.get("completed_games",0)>0 and s.get("elapsed_wall_seconds",0)>0:
    remaining=(s["total_games"]-s["completed_games"])*s["elapsed_wall_seconds"]/s["completed_games"]
    print(f"按当前完成速率估算派发剩余{remaining/60:.1f}分钟；不含后续审计/报告，非保证。")
if (RUN_DIR/"report.html").exists():
    display(FileLink(str(RUN_DIR/"report.html")))
    display(HTML((RUN_DIR/"report.html").read_text("utf-8")))


## 安全停止 / 继续未派发部分
默认先停止派发，让当前分片完成再进入STOPPED_SAFE；之后再调用`start`只跑未派发分片。强停或异常退出的半成品保留且不能静默重跑。以下代码默认不执行，按需取消注释。不要直接中断内核来代替停止。

In [ ]:
# print(arena_rpc("prism_notebook_arena:stop",work=str(RUN_DIR)))
# 等状态变为STOPPED_SAFE后，可继续未派发部分：
# print(arena_rpc("prism_notebook_arena:start",work=str(RUN_DIR)))
# 确实需要强停时：
# print(arena_rpc("prism_notebook_arena:stop",work=str(RUN_DIR),force=True))


## 下载结果与回放
报告包括每对手W/D/L、纯胜/半平配对区间、物理双座、双方VP/速度/购卡0..5VP/颜色折扣/贵族/预留、七动作及完整来源。逐决策/推理耗时没有测量，标NOT_MEASURED。开发结论与正式晋级分开。zip导出放在run目录之外，不覆盖旧下载包。

In [ ]:
# 运行结束后取消注释：
# archive=arena_rpc("prism_notebook_arena:export_zip",work=str(RUN_DIR))
# display(FileLink(archive))
